# Monte Carlo Risk Assessment for Portfolio Management

## Table of Contents

1. Setting up the environment
2. Project configuration
3. Download and prepare the stock data
4. Build the Monte Carlo return simulations
5. Calculate VaR and CVaR
6. Test VaR forecasts against actual returns
7. Run the workflow end-to-end
8. Conclusions

## Project Objective

The main objective of the project is to understand how statistical models with varying stock-return distributions estimate portfolio risk.

The distributions are assessed using Monte Carlo simulations and evaluated through the Value at Risk (VaR) and Conditional Value at Risk (CVaR) metrics of the simulations.

The three distributions of portfolio returns are Multivariate Normal, Multivariate Student-t, and Historical Simulation of stock returns. To evaluate the validity of each distribution and how accurately they estimate portfolio risk, we perform backtests of the process and compare their VaR metrics to the next day's portfolio returns.

## 1. Setting Up the Environment

In [57]:
import datetime as dt
import numpy as np
import pandas as pd
import yfinance as yf

### Importing Libraries
- `datetime (dt)`: For handling date and time operations
- `numpy (np)`: For numerical arrays and linear algebra used in simulations and risk calculations
- `pandas (pd)`: For organizing stock data, returns, and tables in DataFrames and Series
- `yfinance (yf)`: For downloading historical stock-price data from Yahoo Finance

## 2. Project Configuration

The companies being analyzed are defined in the `stocks` variable by their stock tickers, consisting of Apple (`AAPL`), Microsoft(`MSFT`), Google(`GOOG`), and Amazon(`AMZN`). These stocks were selected because they are large and established technology coompanies with substantial trading volume.

The weights, defined by the `weights` variable are equal between stocks (25%). Using equal weights provides a consistent baseline for comparing simulation models without introducing additional weight variables to the portfolio.

The portfolio's starting value (`initial_portfolio`) will have a starting value of $1000, meaning an initial investment of $250 per stock.

In [58]:
stocks = ['AAPL', 'MSFT', 'GOOG', 'AMZN']
weights = [0.25, 0.25, 0.25, 0.25]
initial_portfolio = 1000

## 3. Download and Prepare the Stock Data

#### Defining the Analysis Date Window

To determine the exact timeframe of stock price data being analyzed, an appropriate sample size is required.

The end date (`end_date`) was hardcoded to September 30, 2026, the date the analysis was performed.

A total of 200 trading days will be analyzed in the simulation, meaning a window of 300 total days (`days_back=300`) encapsulates more than 200 trading days in later steps.

Using this information, we can determine the window's start date (`start_date`).

In [59]:
def build_window_dates(days_back=300):
    end_date = dt.datetime(2026, 9, 30) #9/30/2026
    start_date = end_date - dt.timedelta(days=days_back)
    return start_date, end_date


start_date, end_date = build_window_dates(days_back=300)

#### Defining the Stock-Data Function

Now that the start and end dates of the 300-day window are determined, we must get the required stock data for simulations.

Using stock data from Yahoo Finance for each stock in the `stocks` variable, we determine the closing (end of trading day) price for each stock across the final 200 trading days (`trade_days=200`) of the window.

As mentioned previously, using an initial start date (`start`) 300 days before the September 30th 2026 end date (`end`) gives enough space for the 200 trading days being analyzed.

For each stock, the percentage change in price between each trading day is collected in the `returns` variable.

The average percentage daily return for each stock is compiled into the `mean_returns` variable.

We also generate a covarance matrix (`cov_matrix`), which measures the variance of individual stock returns and the relationships between each stock in the portfolio.




In [60]:
def get_data(stocks, start, end, trade_days=200):
    stock_data = yf.download(stocks, start=start, end=end, auto_adjust=False) #download stock data from Yahoo Finance

    if isinstance(stock_data.columns, pd.MultiIndex): #if stock data has multiple columns
        stock_data = stock_data['Close'] #select column labeled 'Close'
    else: #if stock data has 1 column
        if 'Close' in stock_data.columns: #if the 1 column is labeled 'Close'
            stock_data = stock_data[['Close']] #select the column labeled 'Close'
        else: #if the 1 column is not labeled 'Close'
            raise ValueError("The downloaded data does not contain a 'Close' column.") #raise an error

    #stock data is now only closing values for each day in 200-day span

    returns = stock_data.pct_change().tail(trade_days) #compute percentage change in closing value for each day in 200-day span
    mean_returns = returns.mean() #mean of daily returns for the stock
    cov_matrix = returns.cov() #covariance matrix of daily returns for the stock
    return mean_returns, cov_matrix

In [81]:
start_date, end_date = build_window_dates(days_back=300)
mean_returns, cov_matrix = get_data(stocks, start_date, end_date)

print("Mean daily returns (decimal daily return):")
print(mean_returns)
print("\nMean daily returns (%):")
print(mean_returns * 100)
print("\nCovariance matrix (decimal-return units squared):")
print(cov_matrix)

[*********************100%***********************]  4 of 4 completed

Mean daily returns (decimal daily return):
Ticker
AAPL    0.000969
AMZN    0.000537
GOOG    0.000440
MSFT    0.000549
dtype: float64

Mean daily returns (%):
Ticker
AAPL    0.096870
AMZN    0.053659
GOOG    0.043953
MSFT    0.054929
dtype: float64

Covariance matrix (decimal-return units squared):
Ticker      AAPL      AMZN      GOOG      MSFT
Ticker                                        
AAPL    0.000268  0.000019  0.000038  0.000047
AMZN    0.000019  0.000463  0.000260  0.000199
GOOG    0.000038  0.000260  0.000388  0.000076
MSFT    0.000047  0.000199  0.000076  0.000493


## 4. Build the Monte Carlo Return Simulations

In preperation for the Monte Carlo simulations, we must create a mean matrix where each column contains each stock's mean daily return (`mean_returns`), repeated across each column. The number of columns must represent the simulation horizon, also known as the number of trading days being evaluated in each simulation (`num_days`).

This value is set to 250 trading days (approximately 1 year) to generate annual portfolio projections and evaluate longer term portfolio decisions. Since the portfolio is designed for long-term investments, a larger simulation horizon is more applicable than a smaller horizon for high-frequency trading.

In [62]:
num_days = 250


def build_mean_matrix(mean_returns, num_days):
    mean_values = np.asarray(mean_returns) #converting mean_returns to a column vector of shape (n_stocks, 1)
    mean_returns_matrix = np.tile(mean_values.reshape(-1, 1), (1, num_days)) #repeating mean_values column vector num_days times so each column is identical
    return mean_returns_matrix

In [86]:
mean_m = build_mean_matrix(mean_returns, num_days)

### Run Monte Carlo Simulation Using a Multivariate Normal Distribution

When performing a simulation for a Multivariate Normal Distribution, we must generate an array of randomly distributed normal variables for each day of the simulation, represented by $Z_t$ where $t=day$. The random variables follow a standard multivariate normal distribution, meaning all random variables in the array are standard normal.

$$
Z_t \sim \mathcal{N}(0,I)
$$


Then using the covariance matrix of the stocks in the portfolio, we must perform a Cholesky decomposition of the covariance matrix to determine its Cholesky matrix. When the Cholesky matrix ($\Sigma$) is multiplied by its transposed matrix ($L^T$), the result is the covarance matrix ($\Sigma$). 

$$
\Sigma = LL^T
$$


By multiplying the Cholesky matrix ($L$) by the standard normal variables array for a given day ($Z_t$), we can simulate daily returns for stocks that factor in relations between the stocks thenselves. When these variations are added to the average daily returns of each stock ($\mu$), we can estimate the daily returns for each stock for a given day ($R_t$).

$$
R_t = \mu + LZ_t
$$


For each day of the sample ($t$), multiply each stock's simulated daily returns ($R_{i,t}$) by its weight in the portfolio ($w_i$), then add all of these values across each stock ($s$) in the portfolio to get the full portfolio's return for the day ($R_{portfolio,t}$)

$$
R_{portfolio,t}=w_1R_{1,t}+w_2R_{2,t}+...+w_sR_{s,t}
$$


For the first day's ($t=1$) portfolio return ($R_{portfolio,1}$), multiply its compounding factor ($1+R_{portfolio,1}$) by the starting portfolio value ($V_0$) to get the portfolio value for day 1 ($V_{1}$). Each day's new compounding factor ($1+R_{portfolio,t}$) is multiplied by the previous day's portfolio value until the 250th and final day of the simulation ($T$). The value of the portfolio is represented by $V_{1}$

$$
V_{T}=V_0(1+R_{portfolio,1})(1+R_{portfolio,2})...(1+R_{portfolio,T})
$$



Repeat all of these processes 500 times, once for each simulation (`mc_sims=500`) to generate 500 simulated portfolio returns. All of these simulated daily returns are included in the `portfolio_sims` variable.


In [64]:
mc_sims = 500


def simulate_normal_model(mean_m, cov_matrix, weights, initial_portfolio, num_days, mc_sims):
    portfolio_sims = np.full(shape=(num_days, mc_sims), fill_value=0.0) #creates all 0.0 matrix, rows = num_days, columns = mc_sims

    for m in range(mc_sims): #for each column/simulation
        z = np.random.normal(size=(num_days, len(weights))) #array of standard normal vars dimensions (num_days, n_stocks)
        l = np.linalg.cholesky(cov_matrix) #Cholesky decomposition of the covariance matrix comparing stocks to each other
        daily_returns = mean_m + l @ z.T #cholesky matrix*standard normal array + mean returns matrix = daily returns for each stock for each day
        portfolio_sims[:, m] = np.cumprod(daily_returns.T @ weights + 1) * initial_portfolio #for each day, multiply each stock's return by its weight to get the days full portfolio return, repeat for each day

    return portfolio_sims

In [82]:
normal_sims = simulate_normal_model(
    mean_m,
    cov_matrix,
    weights,
    initial_portfolio,
    num_days,
    mc_sims,
)

### Run Monte Carlo Simulation Using a Multivariate Student-t Distribution

The simulation process for a Multivariate Student-t Distribution is similar to the Normal Distribution simulation, with some modifications. These changes will account for the heavier tails of the Student-t distribution, which can increases the chances of extreme stock returns in either direction.

The process of generating an array of randomly distributed standard normal variables ($Z_t$) remains unchanged. However, we must also generate an array of randomly distributed chi-square variables ($u_t$) for a given day ($t$) to transform the standard normal variables into Student-t distributed random variables.

The chi-square variables for each day ($u_t$) will follow a chi-square distribution ($\chi^2_{df}$), where the degrees of freedom ($df$) determine the shape of the Student-t distribution and heaviness of its tails. Lower degrees of freedom result in heavier tails, making the distribution more likely to have extreme returns. The degrees of freedom is set to 8 (`df=8`) because ________________ 

$$
u_t \sim \chi^2_{df}
$$


The chi-square variables are generated using a random number generator (`rng`), allowing us to control the random sampling process. In addition, specifying `rng` to a specific value reproduces the same results across multiple runs.

The Cholesky decomposition process is slightly modified for the Student-t distribution. Since the Student-t distribution has heavier tails and a different variance structure than the standard normal distribution, the covariance matrix ($\Sigma$) must be multiplied by $((df-2)/df)$ before performing the Cholesky decomposition. The resulting lower triangular Cholesky matrix ($L$) and its transpose ($L^T$) produce the scaled covariance matrix. As a result, the Student-t distributed stock returns maintain the same covariance as the Normal Distribution model.

$$
\Sigma \left(\frac{df-2}{df}\right) = LL^T
$$


Next, the standard normal random variables ($Z_t$) are transformed into Student-t distributed random variables ($X_t$) using the formula below.

$$
X_t = \frac{Z_t}{\sqrt{u_t/df}}
$$


Using the modified Cholesky matrix ($L$) and transformed random variables ($X_t$), we can generate simulated daily stock returns by adding the historical average daily returns ($\mu$).

$$
R_t = \mu + LX_t
$$

From this point forward, the process is the same as the Normal Distribution simulation. We multiply each stock's simulated daily returns by its corresponding portfolio weight and sum the weighted returns to calculate the portfolio's daily return. We then compound the daily portfolio returns to calculate the cumulative portfolio value.

The process is repeated for each trading day in the sample to complete 1 simulation. Once 500 Monte Carlo simulation (`mc_sims=500`) are generated. The resulting simulated portfolio values are stored in the `portfolio_sims_t` variable.

In [66]:
def simulate_student_t_model(mean_m, cov_matrix, weights, initial_portfolio, num_days, mc_sims, df=8, rng=None):
    if rng is None: #if no random number generator is provided
        rng = np.random.default_rng() #create default random number generator

    portfolio_sims_t = np.full(shape=(num_days, mc_sims), fill_value=0.0)

    for m in range(mc_sims):
        z = np.random.normal(size=(num_days, len(weights)))
        u = rng.chisquare(df, size=num_days).reshape(num_days, 1) #generate chi-square random variables
        l = np.linalg.cholesky(cov_matrix * (df - 2) / df) #cholesky decomposition scaled by (df-2)/df to account for the Student-t distribution's heavier tails
        x = z / np.sqrt(u / df) #z-term adjusted so the covariance is the same as the multivariate normal distribution
        daily_returns = mean_m + l @ x.T
        portfolio_sims_t[:, m] = np.cumprod(daily_returns.T @ weights + 1) * initial_portfolio

    return portfolio_sims_t

In [84]:
t_sims = simulate_student_t_model(
    mean_m,
    cov_matrix,
    weights,
    initial_portfolio,
    num_days,
    mc_sims,
    df=8,
)

#### Downloading Historical Portfolio Returns

The final simulations will utilize the daily stock returns from 250 randomly selected trading days in the past 5 years (`years_back=5`) before the September 30th 2026 end date.

For each trading day in the 5-year sample size, the return of each stock in the portfolio is calculated and pulled into the `portfolio_returns` variable.

In [68]:
def get_historical_portfolio_returns(stocks, weights, years_back=5):
    end_date = pd.Timestamp("2026-09-30").normalize() #fixed end date: 9/30/2026
    start_date = end_date - pd.DateOffset(years=years_back) #date 5 years before the fixed end date

    prices = yf.download(stocks,start=start_date,end=end_date,auto_adjust=True,)['Close'] #closing prices for each stock on each day in 5-year span

    returns = prices.pct_change().dropna() #percentage change in closing value for each stock and for each day in 5-year span
    portfolio_returns = returns @ weights #multiplies the returns of each stock by its weight, sums returns for each day to get total portfolio return for each day in 5-year span
    return returns, portfolio_returns

In [83]:
historical_stock_returns, portfolio_returns_past_data = get_historical_portfolio_returns(
    stocks,
    weights,
    years_back=5,
)

[*********************100%***********************]  4 of 4 completed


### Sample Historical Return Days with Replacement to Build a Distribution

In one simulation, 250 trading days from the 5-year window are randomly selected .

For each day of the sample, use each stock's weight in the portfolio to get the full portfolio's return for the day. This is the same method as the other models, except with historical stock returns instead of simulated stock returns. The process is repeated for all 250 days and generates a simulated portfolio value for the simulation.

Repeat this process 500 times to generate a daily portfolio return for each simulation (`mc_sims=500`). All of these simulated portfolio returns are compiled into the `portfolio_hist` variable.

In [70]:
def simulate_random_day_model(historical_stock_returns, weights, initial_portfolio, num_days, mc_sims):
    portfolio_hist = np.full(shape=(num_days, mc_sims), fill_value=0.0)

    for m in range(mc_sims):
        sampled_returns = historical_stock_returns.sample(n=num_days, replace=True) #randomly select certain days from the five-year historical returns and use their real returns for each stock
        portfolio_hist[:, m] = np.cumprod(sampled_returns @ weights + 1) * initial_portfolio #randomly selected returns for each stock put together to completely simulate stock returns for each day

    return portfolio_hist

In [85]:
hist_sims = simulate_random_day_model(
    historical_stock_returns,
    weights,
    initial_portfolio,
    num_days,
    mc_sims,
)

## 5. Calculate VaR and CVaR

### Value at Risk: Return the Percentile for a Given Confidence Level

The Value at Risk (VaR) is a way to evaluate lower portfolio return scenarios that are not likely to occur but are still possible. A typical way to measure this is to gather all simulated returns and determine the expected return for a given percentile (represented by alpha). A common percentile to evaluate is the bottom 5% of possible returns (`alpha=5`). This means there is a 5% chance the portfolio will return a value below your Value at Risk when alpha = 5%.

In [72]:
alpha = 5 #5% alpha


def mcValueRisk(returns, alpha=5):
    if isinstance(returns, pd.Series): 
        return np.percentile(returns, alpha) #return the alpha percentile of the returns
    raise TypeError("Expected pandas data series") #raise an error if the input is not a pandas series

### Compute CVaR / Expected Shortfall Below the VaR Threshold

Once Value at Risk (VaR) is generated, it can be used to determine Conditional Value at Risk (CVaR). To determine this, take every simulated portfolio value below the VaR generated before, and determine their mean. This value represents the average portfolio return if the return is anywhere below the 5th percentile of simulations.

In [73]:
def mcConditionalValueRisk(returns, alpha=5):
    if isinstance(returns, pd.Series):
        below_var = returns <= mcValueRisk(returns, alpha=alpha) #generate all returns below alpha percentile of returns
        return returns[below_var].mean() #return mean of all returns below alpha percentile of returns
    raise TypeError("Expected pandas data series")

### Summarize VaR and CVaR for Each Model Column

Using all simulated portfolio returns for all three distribution types (`results_df`), VaR and CVaR are generated and compared across each distribution type.

In [74]:
def summarize_risk(results_df, initial_portfolio, alpha=5):
    VaR = results_df.apply(lambda col: initial_portfolio - mcValueRisk(col, alpha=alpha)) #for each distribution type, determine VaR by subtracting alpha percentile final return from initial portfolio value
    CondVaR = results_df.apply(lambda col: initial_portfolio - mcConditionalValueRisk(col, alpha=alpha)) #for each distribution type, determine CVaR by subtracting mean of returns below alpha percentile from initial portfolio value
    return pd.DataFrame({'VaR': VaR, 'CVaR': CondVaR})

In [75]:
results_df = pd.DataFrame({
    'normal': normal_sims[-1, :],
    't': t_sims[-1, :],
    'hist': hist_sims[-1, :],
})

risk_summary = summarize_risk(results_df, initial_portfolio, alpha=alpha)
print("Risk Summary")
print(risk_summary)

Risk Summary
               VaR        CVaR
normal  105.052190  129.728732
t       135.622631  165.745663
hist    130.536409  164.992110


## 6. Test VaR Forecasts Against Actual Returns

### Backtest VaR Models Against Realized Portfolio Returns

Backtesting is used to test if the VaR and CVaR forecasts would be accurate if used in practice.

In this example, we will use rolling historical data that uses 250-day windows (`window_backtest=250`) to evaluate the immediate proceeding trade day's return.  

Example:  

| Backtest Window | Day Evaluated | 
| --- | --- | 
| Days 1-250 | Day 251| 
| Days 2-251| Day 252 | 
| Days 3-252| Day 253 |  
| ...| ...|  


For each day, individual `stock_returns` and overall `portfolio_returns` over the 5-year historical window are collected and stores as `backtest_data` for testing (the first 250 days of historical data can't be tested since it doesn't have enough previous days to create a 250-day window).

For the first 250-day backtest window of historical data (to test day 251), calculate the average return for each stock (`mu`) and its covaraince matrix (`cov`) for the stocks. In addition, calculate the returns for the full portfolio across each day of the window, and perform a Cholesky decomposition to derive `L` from the `cov` matrix.

Using the same procedure as the standard normal distribution, repeat the initial Monte Carlo simulation process across the 250-day backtest window. Repeat this procedure for the t-distribution and historical data simulations too, and all returns will be stored in `backtest_data`.

Simulate each process 10000 times (`num_sims_backtest=10000`), and the 5th percentile (`alpha_backtest=5`) of the backtest returns is the distribution's VaR.
NOTE: 10000 simulations are used because a larger 250-day sample size requires a larger number of simulations for a more precise VaR calculation.

Repeat the complete simulation process for the next 250-day backtest window (days 2-251 to test day 252) to derive new VaR values for each simulation type. This will continue 1 trading day at a time until the final trading day of the 5-year historical window is tested.

Violation rate: Violation rate is a way to evaluate the percentage of the time that the portfolio's return is less than the VaR calculated using the preceeding backtest. Since VaR is calculated using an alpha value of 5%, it is expected for the violation rate to be 5% (since the previous 250 days simulated 5% of returns to be below that value). A higher rate means the backtest VaR underestimated the next day's risk, while a lower rate means the backtestVaR overestimated risk.  

For the normal distrubution simulations and for each day being tested, if the actual return of the portfolio is less than the VaR derived from the previous 250 days, that is considered a violation. This process is repeated for each day where a backtest was performed, and the percentage of total days where a violation is performed represents the violation rate of the distribution type. This process is repeated for the t-distrubiton and historical data VaR values.

In [76]:
def run_var_backtest(stock_returns, portfolio_returns, weights, window_backtest=250, alpha_backtest=5, num_sims_backtest=10000, df_backtest=8, rng=None):
    if rng is None:  # if no rng is provided
        rng = np.random.default_rng(10)  # create new rng

    backtest_data = pd.DataFrame(
        index=stock_returns.index[window_backtest:],  # index by date, only uses dates after the initial window_backtest period
        columns=['Normal', 'Student-t', 'Historical'],
        dtype=float,
    )
    backtest_data['Actual Return'] = portfolio_returns.iloc[window_backtest:].to_numpy()  # adds actual portfolio returns for each day in the backtest period

    stocks = stock_returns.columns  # stocks are the same as the column names in the stock-returns DataFrame

    for i in range(window_backtest, len(stock_returns)):  # for each day in the backtest period

        #general info
        historical_window = stock_returns.iloc[i - window_backtest:i]  # historical window of stock returns is 250 days before the current day being evaluated
        mu = historical_window.mean().to_numpy()  # average return for each stock over the historical window
        cov = historical_window.cov().to_numpy()  # covariance matrix of returns for each stock over the historical window

        #normal backtest
        z_normal = rng.standard_normal(size=(num_sims_backtest, len(stocks)))  # standard normal random variables for each stock for each simulation
        L = np.linalg.cholesky(cov)  # Cholesky decomposition of the covariance matrix
        simulated_normal = mu + z_normal @ L.T  # simulate daily returns for each stock
        portfolio_normal = simulated_normal @ weights  # generates full portfolio returns for each day
        backtest_data.iloc[i - window_backtest, 0] = np.percentile(portfolio_normal, alpha_backtest)  # calculates VaR (bottom alpha percentile) out of the simulated portfolio end returns

        #t backtest
        z_t = rng.standard_normal(size=(num_sims_backtest, len(stocks)))  # standard normal random variables for each stock for each simulation
        u_t = rng.chisquare(df_backtest, size=num_sims_backtest)  # chi-square random variables for each simulation
        L_t = np.linalg.cholesky(cov * (df_backtest - 2) / df_backtest)  # cholesky decomposition scaled by (df-2)/df to account for the Student-t distribution's heavier tails
        x_t = z_t / np.sqrt(u_t / df_backtest)[:, None]  # z-term adjusted so the covariance is the same as the multivariate normal distribution
        simulated_t = mu + x_t @ L_t.T
        portfolio_t = simulated_t @ weights  # full-portfolio returns for each day
        backtest_data.iloc[i - window_backtest, 1] = np.percentile(portfolio_t, alpha_backtest)  # VaR of t-test

        #historical backtest
        historical_portfolio = historical_window @ weights  # full portfolio returns for each day
        backtest_data.iloc[i - window_backtest, 2] = np.percentile(historical_portfolio, alpha_backtest)  # VaR of actual data for each day in the backtest period

    for model in ['Normal', 'Student-t', 'Historical']:  # for each model type
        backtest_data[f'{model} Violation (T/F)'] = (backtest_data['Actual Return'] < backtest_data[model])  # if the actual return is less than the VaR for that day, it is a violation

    violation_rates = backtest_data.iloc[:, 4:7].mean().mul(100)
    summary = pd.DataFrame({
        'Model': ['Normal', 'Student-t', 'Historical'],
        'Expected Violations (%)': [5.0, 5.0, 5.0],
        'Actual Violations (%)': violation_rates.values,  # percentage of True's in each column (% of days actual returns < VaR)
    })

    return backtest_data, summary

In [77]:
backtest_data, backtest_summary = run_var_backtest(
    historical_stock_returns,
    portfolio_returns_past_data,
    weights,
    window_backtest=250,
    alpha_backtest=5,
    num_sims_backtest=10000,
    df_backtest=8,
)

print("VaR Backtest Summary")
print(backtest_summary)

VaR Backtest Summary
        Model  Expected Violations (%)  Actual Violations (%)
0      Normal                      5.0               4.586241
1   Student-t                      5.0               4.685942
2  Historical                      5.0               4.287139


#### Defining Backtesting Parameters:

- `df_backtest`: Degrees of freedom for the Student-t backtesting model

ANALYZE RESULTS ______________

## 7. Run the Workflow End-to-End

The `main` function encapsulates all programming run in this file. 

In [78]:
def main():
    start_date, end_date = build_window_dates(days_back=300)
    mean_returns, cov_matrix = get_data(stocks, start_date, end_date) #uses 200 recent trading days for normal and Student-t parameter estimates
    mean_m = build_mean_matrix(mean_returns, num_days) #mean_returns column vector repeated num_days times

    historical_stock_returns, portfolio_returns_past_data = get_historical_portfolio_returns(
        stocks,
        weights,
        years_back=5,
    )

    normal_sims = simulate_normal_model(mean_m, cov_matrix, weights, initial_portfolio, num_days, mc_sims) #simulates full portfolio returns for each day
    t_sims = simulate_student_t_model(mean_m, cov_matrix, weights, initial_portfolio, num_days, mc_sims, df=8)
    hist_sims = simulate_random_day_model(historical_stock_returns, weights, initial_portfolio, num_days, mc_sims)

    results_df = pd.DataFrame({ #creates dataframe with the final portfolio value for each simulation for each model
        'normal': normal_sims[-1, :],
        't': t_sims[-1, :],
        'hist': hist_sims[-1, :],
    })

    risk_summary = summarize_risk(results_df, initial_portfolio, alpha=alpha) #generate VaR and CVaR for each model based on final portfolio returns
    print('Risk Summary')
    print(risk_summary)

    backtest_data, summary = run_var_backtest(
        historical_stock_returns, #returns of each stock each day in 5-year window
        portfolio_returns_past_data, #returns of portfolio as a whole each day in 5-year window
        weights,
        window_backtest=250, #number of days to use for historical data to calculate VaR
        alpha_backtest=5,
        num_sims_backtest=10000,
        df_backtest=8, #same df as used in the Student-t model
    )

    print('\nVaR Backtest Summary')
    print(summary)

In [ ]:
# Optional end-to-end rerun:
main()

## 8. Conclusions

Descriptions of findings here ______________


### Next Steps:

- Analyzing how results vary for stocks of companies in other industries or sizes (rather than large technology companies)
- The effects of changing weights of stocks in the portfolio (rather than equal amongst each company)
- A larger or smaller simulation horizon for the Monte Carlo simulations to evaluate VaR for portfolio decisions longer or shorter than 1 year. 